# Phase 9.7: Diverse Dataset Injection → Flux-capable.flx

**Creating a Capable FLUX Model via Physics-Based Learning**

This notebook injects **~155k samples** from diverse high-quality datasets into the FLUX field using training-free physics-based injection.

## Memory-Safe Features
- **All counts HALVED** for Kaggle T4 (16GB VRAM, 13GB RAM)
- **Memory clearing** after each dataset
- **CPU-based settling** when VRAM is low
- **Streaming** for large datasets (FLAN)

## Key Difference from Phase 9.6
- Properly loads and preserves ALL components from Flux-X-complete.flx
- Injects knowledge INTO the existing field (additive)
- Saves a complete .flx with ALL 14 components intact

## Datasets (HALVED for Memory Safety)

| Dataset | Size | Domain |
|---------|------|--------|
| **SlimOrca** | 25k | High-quality instruction following |
| **OpenHermes** | 25k | Curated quality responses |
| **FLAN** | 25k | Diverse task templates (streamed) |
| **UltraChat** | 15k | High-quality conversations |
| **MathInstruct** | 15k | Mathematical reasoning |
| **Evol-Instruct** | 15k | Complex evolved instructions |
| OpenAssistant | 5k | Multi-turn assistancy |
| Dolly | 7.5k | Natural dialogue |
| GSM8K | 4k | Math chain-of-thought |
| Capybara | 8k | Multi-turn conversations |
| CodeAlpaca | 10k | Programming tasks |

**Total: ~155k samples** (HALVED for memory safety)

## Expected Runtime
- ~20-30 min on T4 GPU
- RAM usage: Peak ~5GB, steady ~3GB
- Output: `Flux-capable.flx` (~2.1 GB)

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets transformers

print('  ✓ Dependencies installed')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
ROOT_DIR = ROOT
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for i in range(1, 10):
    p = PHASES_DIR / f'phase{i}'
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Add phase subdirs
for subdir in ['phase8_8', 'phase8_9', 'phase9']:
    p = PHASES_DIR / subdir
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print(f'  ✓ {subdir} found')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=9)  # Log to phase9.log
log.separator("Phase 9.7: MASSIVE Dataset Injection")

print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load secrets (Kaggle → Colab → env fallback)
hf_token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Fall back to env vars
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment variables loaded')

# Clean and set token
if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token
    print('  ✓ HF_TOKEN set')
else:
    print('  ⚠ HF_TOKEN not found (will try anonymous download)')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download & Load Flux-X-complete.flx

We load the most capable base model and will inject knowledge INTO it.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

log.cell_start("Cell 4 — Download & Load FLUX Model")

# Must use Flux-X-complete.flx - our most capable model
FLX_NAME = 'Flux-X-complete.flx'
FLX_HF_PATH = 'checkpoints/Flux-X-complete.flx'

flx_path = CHECKPOINT_DIR / FLX_NAME

# Check local first
if flx_path.exists():
    size_mb = flx_path.stat().st_size / 1e6
    print(f'  ✓ Found local {FLX_NAME} ({size_mb:.1f} MB)')
else:
    print(f'  Downloading {FLX_NAME}...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename=FLX_HF_PATH,
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        flx_path = CHECKPOINT_DIR / FLX_NAME
        size_mb = Path(downloaded).stat().st_size / 1e6
        print(f'  ✓ Downloaded {FLX_NAME} ({size_mb:.1f} MB)')
    except Exception as e:
        raise RuntimeError(f"Cannot download {FLX_NAME}: {e}")

# Load the model
try:
    from flx_loader import verify_flx, get_flx_info, load_flux_from_flx
    print('  ✓ flx_loader imported')
except ImportError as e:
    print(f'  ⚠ Import error: {e}')
    raise

# Verify the archive
success, msg = verify_flx(flx_path)
print(f'  Verification: {msg}')

if not success:
    raise RuntimeError(f"FLX verification failed: {msg}")

# Show info
info = get_flx_info(flx_path)
print(f'  Version: {info["version"]}')

# Load the full model
model = load_flux_from_flx(
    path=flx_path,
    device=device,
    verbose=True,
    auto_download=False,
)

# Store original .flx for later merging
print('\n  Loading original .flx archive for component preservation...')
original_flx = torch.load(str(flx_path), map_location='cpu', weights_only=False)
print(f'  ✓ Original .flx loaded ({len(original_flx)} top-level keys)')
print(f'    Keys: {list(original_flx.keys())}')

# Quick stats
stats = model.get_stats()
log.metric("Total params", f"{stats.total_params:,}")
log.metric("Field energy", f"{stats.field_energy:.2f}")

log.cell_end("Cell 4 — Download & Load FLUX Model", "PASS")

## Cell 5: Load Diverse Datasets (Memory-Safe - HALVED)

**Kaggle-Safe Loading**: All counts halved for memory safety.

| Dataset | Target Size | Purpose |
|---------|-------------|--------|
| SlimOrca | 25k | Instruction following |
| UltraChat | 15k | Quality conversations |
| OpenHermes | 25k | Quality responses |
| FLAN | 25k | Task templates |
| MathInstruct | 15k | Math reasoning |
| Evol-Instruct | 15k | Complex instructions |
| OpenAssistant | 5k | Assistancy |
| Dolly | 7.5k | Dialogue |
| GSM8K | 4k | Chain-of-thought |
| Capybara | 8k | Multi-turn |
| CodeAlpaca | 10k | Programming |

**Total: ~155k samples** (HALVED for Kaggle memory safety)

In [ ]:
log.cell_start("Cell 5 — Load Diverse Datasets (Memory-Safe - HALVED)")

from datasets import load_dataset
import random
import gc

print("\n  Loading diverse datasets (HALVED FOR MEMORY SAFETY)...")
print("  " + "="*60)
print("  Target: ~155k samples (all counts halved)")
print("  Strategy: Load → Extract → Clear → Next")
print("  " + "="*60)

datasets_loaded = {}

def clear_memory():
    """Clear Python and CUDA memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────
# ALL COUNTS HALVED FOR MEMORY SAFETY
# ─────────────────────────────────────────────

# 1. SlimOrca: 50k → 25k
print("\n  [1/11] SlimOrca (instructions)...")
try:
    slimorca = load_dataset("Open-Orca/SlimOrca", split="train")
    slimorca_texts = []
    for item in slimorca:
        if 'conversations' in item:
            conv = item['conversations']
            if isinstance(conv, list):
                text = "\n".join([f"{turn.get('from', 'user')}: {turn.get('value', '')}" 
                                  for turn in conv if isinstance(turn, dict)])
                if len(text) > 50:
                    slimorca_texts.append(text)
    datasets_loaded['slimorca'] = random.sample(slimorca_texts, min(25000, len(slimorca_texts)))
    del slimorca, slimorca_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['slimorca']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {str(e)[:50]}")
    datasets_loaded['slimorca'] = []

# 2. UltraChat: 30k → 15k
print("\n  [2/11] UltraChat (conversations)...")
try:
    ultrachat = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
    ultrachat_texts = []
    for item in ultrachat:
        if 'messages' in item:
            msgs = item['messages']
            if isinstance(msgs, list):
                text = "\n".join([f"{msg.get('role', 'user')}: {msg.get('content', '')}" 
                                  for msg in msgs if isinstance(msg, dict)])
                if len(text) > 50:
                    ultrachat_texts.append(text)
    datasets_loaded['ultrachat'] = random.sample(ultrachat_texts, min(15000, len(ultrachat_texts)))
    del ultrachat, ultrachat_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['ultrachat']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {str(e)[:50]}")
    datasets_loaded['ultrachat'] = []

# 3. OpenHermes: 50k → 25k
print("\n  [3/11] OpenHermes (quality responses)...")
try:
    openhermes = load_dataset("teknium/OpenHermes-2.5", split="train")
    oh_texts = []
    for item in openhermes:
        if 'conversations' in item:
            conv = item['conversations']
            if isinstance(conv, list):
                text = "\n".join([f"{turn.get('from', 'user')}: {turn.get('value', '')}" 
                                  for turn in conv if isinstance(turn, dict)])
                if len(text) > 50:
                    oh_texts.append(text)
    datasets_loaded['openhermes'] = random.sample(oh_texts, min(25000, len(oh_texts)))
    del openhermes, oh_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['openhermes']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {str(e)[:50]}")
    datasets_loaded['openhermes'] = []

# 4. MathInstruct: 30k → 15k
print("\n  [4/11] MathInstruct (math reasoning)...")
try:
    mathinst = load_dataset("TIGER-Lab/MathInstruct", split="train")
    math_texts = []
    for item in mathinst:
        instruction = item.get('instruction', '')
        output = item.get('output', '')
        if instruction and output:
            text = f"Problem: {instruction}\n\nSolution: {output}"
            math_texts.append(text)
    datasets_loaded['mathinst'] = random.sample(math_texts, min(15000, len(math_texts)))
    del mathinst, math_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['mathinst']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {str(e)[:50]}")
    datasets_loaded['mathinst'] = []

# 5. Evol-Instruct: 30k → 15k
print("\n  [5/11] Evol-Instruct (complex instructions)...")
try:
    evol = load_dataset("WizardLM/WizardLM_evol_instruct_V2_196k", split="train")
    evol_texts = []
    for item in evol:
        if 'conversations' in item:
            conv = item['conversations']
            if isinstance(conv, list):
                text = "\n".join([f"{turn.get('from', 'human')}: {turn.get('value', '')}" 
                                  for turn in conv if isinstance(turn, dict)])
                if len(text) > 50:
                    evol_texts.append(text)
    datasets_loaded['evol'] = random.sample(evol_texts, min(15000, len(evol_texts)))
    del evol, evol_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['evol']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {str(e)[:50]}")
    datasets_loaded['evol'] = []

# 6. FLAN: 50k → 25k (streaming)
print("\n  [6/11] FLAN (task templates, streaming)...")
try:
    flan = load_dataset("Muennighoff/flan", split="train", streaming=True)
    flan_texts = []
    for i, item in enumerate(flan):
        if i >= 25000:  # HALVED
            break
        if 'inputs' in item and 'targets' in item:
            text = f"Task: {item['inputs']}\nAnswer: {item['targets']}"
            flan_texts.append(text)
    datasets_loaded['flan'] = flan_texts
    del flan
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['flan']):,} samples (streamed)")
except Exception as e:
    print(f"       ⚠ Failed: {str(e)[:50]}")
    datasets_loaded['flan'] = []

# ─────────────────────────────────────────────
# ORIGINAL DATASETS (HALVED)
# ─────────────────────────────────────────────

# 7. OpenAssistant: 10k → 5k
print("\n  [7/11] OpenAssistant...")
try:
    oasst = load_dataset("OpenAssistant/oasst1", split="train")
    oasst_texts = [item['text'] for item in oasst if item.get('text')]
    datasets_loaded['assistancy'] = oasst_texts[:5000]  # HALVED
    del oasst, oasst_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['assistancy']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['assistancy'] = []

# 8. Dolly: 15k → 7.5k
print("\n  [8/11] Dolly-15k Dialogue...")
try:
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
    dolly_texts = []
    for item in dolly:
        text = f"User: {item['instruction']}"
        if item.get('context'):
            text += f"\nContext: {item['context']}"
        text += f"\nAssistant: {item['response']}"
        dolly_texts.append(text)
    datasets_loaded['dialogue'] = dolly_texts[:7500]  # HALVED
    del dolly
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['dialogue']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['dialogue'] = []

# 9. GSM8K: 8k → 4k
print("\n  [9/11] GSM8K Math Reasoning...")
try:
    gsm8k = load_dataset("gsm8k", "main", split="train")
    gsm_texts = [f"Problem: {item['question']}\n\nSolution: {item['answer']}" for item in gsm8k]
    datasets_loaded['reasoning'] = gsm_texts[:4000]  # HALVED
    del gsm8k
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['reasoning']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['reasoning'] = []

# 10. Capybara: 16k → 8k
print("\n  [10/11] Capybara Multi-turn...")
try:
    capybara = load_dataset("LDJnr/Capybara", split="train")
    capy_texts = []
    for item in capybara:
        if 'conversation' in item:
            conv = item['conversation']
            if isinstance(conv, list):
                text = "\n".join([f"{turn.get('role', 'user')}: {turn.get('content', '')}" 
                                  for turn in conv if isinstance(turn, dict)])
                capy_texts.append(text)
    datasets_loaded['multiturn'] = capy_texts[:8000]  # HALVED
    del capybara, capy_texts
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['multiturn']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['multiturn'] = []

# 11. Code Alpaca: 20k → 10k
print("\n  [11/11] Code Alpaca Programming...")
try:
    code = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    code_texts = []
    for item in code:
        text = f"Task: {item.get('instruction', '')}"
        if item.get('input'):
            text += f"\nInput: {item['input']}"
        text += f"\nCode: {item.get('output', '')}"
        code_texts.append(text)
    datasets_loaded['code'] = code_texts[:10000]  # HALVED
    del code
    clear_memory()
    print(f"       ✓ {len(datasets_loaded['code']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['code'] = []

# ─────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────
print("\n  " + "="*60)
print("  DATASET SUMMARY (ALL COUNTS HALVED):")
total_samples = 0
for domain, texts in datasets_loaded.items():
    count = len(texts)
    total_samples += count
    status = "✓" if count > 0 else "✗"
    print(f"    {status} {domain:15s}: {count:>8,} samples")
print(f"  {'─'*40}")
print(f"    {'TOTAL':15s}: {total_samples:>8,} samples")
print("  " + "="*60)

# Final memory status
try:
    import psutil
    ram_gb = psutil.virtual_memory().available / 1e9
    print(f"\n  RAM available: {ram_gb:.1f} GB")
except:
    pass

log.metric("Total samples", f"{total_samples:,}")
log.metric("Domains loaded", len([d for d in datasets_loaded if datasets_loaded[d]]))

log.cell_end("Cell 5 — Load Diverse Datasets (Memory-Safe - HALVED)", "PASS")

## Cell 6: Training-Free Injection Framework

In [ ]:
log.cell_start("Cell 6 — Training-Free Injection Framework")

import hashlib
import torch
import time

# ─────────────────────────────────────────────
# Configuration (tuned for massive injection)
# ─────────────────────────────────────────────

INJECTION_SCALE = 0.08     # Slightly lower for massive data
MASS_PER_SAMPLE = 1.0
SETTLE_EVERY = 2000        # More frequent settling for large data
SETTLE_STEPS = 3
MAX_TEXT_LENGTH = 512
INJECTION_RADIUS = 1

print("\n  Training-Free Injection Framework (MASSIVE SCALE)")
print("  " + "="*50)
print(f"    Injection scale: {INJECTION_SCALE}")
print(f"    Settle every: {SETTLE_EVERY} samples")
print(f"    Max text length: {MAX_TEXT_LENGTH} chars")

# ─────────────────────────────────────────────
# Core functions
# ─────────────────────────────────────────────

def text_to_field_coords(text: str, cse, field, use_hash: bool = True):
    """Convert text → field coordinates using CSE + hash."""
    H, W, D = field.h, field.w, field.d
    
    wave = cse.encode(text[:MAX_TEXT_LENGTH])
    
    if hasattr(wave, 'full'):
        wave_vec = wave.full.mean(dim=0)
    elif hasattr(wave, 'semantic'):
        wave_vec = wave.semantic.mean(dim=0)
    else:
        wave_vec = wave.mean(dim=0) if wave.dim() > 1 else wave
    
    wave_vec = wave_vec.cpu()
    
    if use_hash:
        h_hash = hashlib.md5(text.encode()).hexdigest()
        base_h = int(h_hash[:8], 16) % H
        base_w = int(h_hash[8:16], 16) % W
        base_d = int(h_hash[16:24], 16) % D
        
        off_h = int((wave_vec[0].item() * 100) % 5) - 2
        off_w = int((wave_vec[100 % len(wave_vec)].item() * 100) % 5) - 2
        off_d = int((wave_vec[200 % len(wave_vec)].item() * 100) % 5) - 2
        
        h = max(0, min(H - 1, base_h + off_h))
        w = max(0, min(W - 1, base_w + off_w))
        d = max(0, min(D - 1, base_d + off_d))
    else:
        h = int((wave_vec[0].item() + 1) * (H - 1) / 2) % H
        w = int((wave_vec[100 % len(wave_vec)].item() + 1) * (W - 1) / 2) % W
        d = int((wave_vec[200 % len(wave_vec)].item() + 1) * (D - 1) / 2) % D
    
    features = torch.zeros(field.features)
    copy_len = min(len(wave_vec), field.features)
    features[:copy_len] = wave_vec[:copy_len]
    
    return (h, w, d), features


def inject_into_field(field, coords, features, scale=0.1, mass=1.0, radius=1):
    """Inject semantic features into field."""
    h, w, d = coords
    H, W, D = field.h, field.w, field.d
    
    h_slice = slice(max(0, h - radius), min(H, h + radius + 1))
    w_slice = slice(max(0, w - radius), min(W, w + radius + 1))
    d_slice = slice(max(0, d - radius), min(D, d + radius + 1))
    
    injection = features.to(field.state.device)
    field.state[h_slice, w_slice, d_slice] += injection * scale
    field.mass[h, w, d] += mass
    field.energy[h_slice, w_slice, d_slice] *= 0.98


def thermodynamic_settle(field, steps=3, force_cpu=False):
    """
    Settle field to thermodynamic equilibrium.
    Memory-efficient version that works on CPU if needed.
    """
    with torch.no_grad():
        original_device = field.state.device
        
        # Check if we should use CPU (memory-safe)
        use_cpu = force_cpu
        if not use_cpu and torch.cuda.is_available():
            free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
            # Need ~3.4GB for clone operation, use CPU if less than 4GB free
            if free_mem < 4e9:
                use_cpu = True
                print("    (settling on CPU - low VRAM)")
        
        if use_cpu:
            # Move to CPU for settling
            state_cpu = field.state.cpu()
            H, W, D, F = state_cpu.shape
            
            for _ in range(steps):
                # In-place averaging on CPU (no clone needed)
                # Simple exponential smoothing
                state_cpu *= 0.95
                
                # Add neighbor influence (in-place)
                state_cpu[1:, :, :, :] += 0.05/6 * state_cpu[:-1, :, :, :].clone()
                state_cpu[:-1, :, :, :] += 0.05/6 * state_cpu[1:, :, :, :].clone()
                state_cpu[:, 1:, :, :] += 0.05/6 * state_cpu[:, :-1, :, :].clone()
                state_cpu[:, :-1, :, :] += 0.05/6 * state_cpu[:, 1:, :, :].clone()
                state_cpu[:, :, 1:, :] += 0.05/6 * state_cpu[:, :, :-1, :].clone()
                state_cpu[:, :, :-1, :] += 0.05/6 * state_cpu[:, :, 1:, :].clone()
            
            # Move back to original device
            field.state.copy_(state_cpu.to(original_device))
            del state_cpu
            
        else:
            # Original GPU version
            H, W, D, F = field.state.shape
            
            for _ in range(steps):
                settled = field.state.clone()
                
                settled[1:, :, :, :] += field.state[:-1, :, :, :]
                settled[:-1, :, :, :] += field.state[1:, :, :, :]
                settled[:, 1:, :, :] += field.state[:, :-1, :, :]
                settled[:, :-1, :, :] += field.state[:, 1:, :, :]
                settled[:, :, 1:, :] += field.state[:, :, :-1, :]
                settled[:, :, :-1, :] += field.state[:, :, 1:, :]
                
                neighbor_count = torch.ones(H, W, D, device=field.state.device) * 7
                neighbor_count[0, :, :] -= 1
                neighbor_count[-1, :, :] -= 1
                neighbor_count[:, 0, :] -= 1
                neighbor_count[:, -1, :] -= 1
                neighbor_count[:, :, 0] -= 1
                neighbor_count[:, :, -1] -= 1
                
                settled = settled / neighbor_count.unsqueeze(-1)
                field.state = 0.95 * field.state + 0.05 * settled
                
                del settled, neighbor_count
        
        field.energy *= 0.99
        
        # Clear CUDA cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("  ✓ Injection framework defined (memory-safe settling)")
log.cell_end("Cell 6 — Training-Free Injection Framework", "PASS")

## Cell 7: MASSIVE Injection into Field (with Checkpoints)

Injecting ALL datasets into the field with **intermediate saves** every 100k samples for safety.

In [ ]:
log.cell_start("Cell 7 — MASSIVE Injection into Field (with Checkpoints)")

import time
from collections import defaultdict
from datetime import datetime

print("\n  Starting MASSIVE Training-Free Dataset Injection")
print("  " + "="*60)
print("  NO GRADIENTS. NO BACKPROP. JUST PHYSICS.")
print("  " + "="*60)
print(f"\n  Total samples to inject: {total_samples:,}")
print(f"  Estimated time: {total_samples / 200 / 60:.0f}-{total_samples / 150 / 60:.0f} minutes")
print(f"  Intermediate checkpoints: every 100k samples")

# Injection parameters
INJECTION_SCALE = 0.8
MASS_PER_SAMPLE = 0.1
INJECTION_RADIUS = 2
SETTLE_EVERY = 2000
SETTLE_STEPS = 3
CHECKPOINT_INTERVAL = 100000

# Domain-specific scales (emphasize quality sources)
DOMAIN_SCALES = {
    'slimorca': 0.9,
    'ultrachat': 0.8,
    'openhermes': 0.9,
    'math': 1.0,
    'evol': 0.8,
    'flan': 0.7,
    'oasst': 0.9,
    'dolly': 0.8,
    'gsm8k': 1.0,
    'capybara': 0.85,
    'code': 0.9,
}

# Stats tracking
injection_stats = defaultdict(lambda: {'count': 0, 'failed': 0})
total_injected = 0
total_failed = 0
start_time = time.time()
last_checkpoint = 0

# Track spatial distribution
location_histogram = torch.zeros(model.field.h, model.field.w, model.field.d)

# Pre-injection field stats
pre_stats = {
    'mass_sum': model.field.mass.sum().item(),
    'state_norm': model.field.state.norm().item(),
    'energy_mean': model.field.energy.mean().item(),
}
print(f"\n  Pre-injection field state:")
print(f"    Mass sum: {pre_stats['mass_sum']:.4f}")
print(f"    State norm: {pre_stats['state_norm']:.4f}")
print(f"    Energy mean: {pre_stats['energy_mean']:.6f}")

# Intermediate checkpoint helper
def save_intermediate_checkpoint(name, samples):
    """Save intermediate checkpoint to recover from crashes."""
    try:
        intermediate = {
            'field_state': model.field.state.cpu(),
            'field_mass': model.field.mass.cpu(),
            'field_energy': model.field.energy.cpu(),
            'samples_injected': samples,
            'injection_stats': dict(injection_stats),
            'timestamp': datetime.now().isoformat(),
        }
        path = Path('/kaggle/working') / name
        torch.save(intermediate, path)
        print(f"\n    ✓ Checkpoint saved: {name} ({samples:,} samples)")
        del intermediate
        gc.collect()
    except Exception as e:
        print(f"    ⚠ Checkpoint failed: {e}")

print(f"\n  Injecting datasets...")

for domain, texts in datasets_loaded.items():
    if not texts:
        continue
    
    domain_scale = DOMAIN_SCALES.get(domain, INJECTION_SCALE)
    print(f"\n  [{domain.upper()}] {len(texts):,} samples (scale={domain_scale})")
    
    domain_start = time.time()
    
    for i, text in enumerate(texts):
        if not text or len(text.strip()) < 10:
            injection_stats[domain]['failed'] += 1
            total_failed += 1
            continue
        
        try:
            coords, features = text_to_field_coords(text, model.cse, model.field)
            inject_into_field(
                model.field, coords, features,
                scale=domain_scale, mass=MASS_PER_SAMPLE, radius=INJECTION_RADIUS
            )
            
            h, w, d = coords
            location_histogram[h, w, d] += 1
            
            injection_stats[domain]['count'] += 1
            total_injected += 1
            
        except Exception as e:
            injection_stats[domain]['failed'] += 1
            total_failed += 1
            continue
        
        # Progress + settling
        if (total_injected + 1) % SETTLE_EVERY == 0:
            thermodynamic_settle(model.field, steps=SETTLE_STEPS)
            elapsed = time.time() - start_time
            rate = total_injected / elapsed
            eta = (total_samples - total_injected) / rate / 60
            print(f"    ... {total_injected:,}/{total_samples:,} ({rate:.0f}/sec) — ETA: {eta:.1f}min")
        
        # Intermediate checkpoint every 100k
        if total_injected - last_checkpoint >= CHECKPOINT_INTERVAL:
            save_intermediate_checkpoint(f'Flux-capable-{total_injected//1000}k.pt', total_injected)
            last_checkpoint = total_injected
    
    domain_elapsed = time.time() - domain_start
    print(f"    ✓ {injection_stats[domain]['count']:,} injected in {domain_elapsed:.1f}s")
    
    # Clear domain texts from memory after processing
    datasets_loaded[domain] = []
    gc.collect()

# SKIP final settling - we already settled every 2000 samples
# Combined: 155k / 2000 = ~77 settling passes already done
print(f"\n  ✓ Skipping final settling (77+ settling passes already done during injection)")

# Clear GPU cache for save
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

total_time = time.time() - start_time

# Post-injection stats
post_stats = {
    'mass_sum': model.field.mass.sum().item(),
    'state_norm': model.field.state.norm().item(),
    'energy_mean': model.field.energy.mean().item(),
}

unique_locations = (location_histogram > 0).sum().item()
max_overlap = location_histogram.max().item()

print(f"\n  " + "="*60)
print(f"  ✓ MASSIVE INJECTION COMPLETE")
print(f"  " + "="*60)
print(f"\n  Injection summary:")
print(f"    Total injected: {total_injected:,}")
print(f"    Total failed: {total_failed:,}")
print(f"    Success rate: {100*total_injected/(total_injected+total_failed):.1f}%")
print(f"    Time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"    Rate: {total_injected/total_time:.0f} samples/sec")

print(f"\n  Field transformation:")
print(f"    Mass: {pre_stats['mass_sum']:.2f} → {post_stats['mass_sum']:.2f}")
print(f"    State norm: {pre_stats['state_norm']:.2f} → {post_stats['state_norm']:.2f}")
print(f"    Energy: {pre_stats['energy_mean']:.6f} → {post_stats['energy_mean']:.6f}")

print(f"\n  Spatial coverage:")
print(f"    Unique locations: {unique_locations:,} / {model.field.h * model.field.w * model.field.d:,}")
print(f"    Coverage: {100*unique_locations/(model.field.h * model.field.w * model.field.d):.1f}%")
print(f"    Max overlap: {max_overlap:.0f} samples at one location")

print(f"\n  Per-domain breakdown:")
for domain, stats in sorted(injection_stats.items()):
    print(f"    {domain}: {stats['count']:,} injected, {stats['failed']} failed")

log.cell_end("Cell 7 — MASSIVE Injection into Field (with Checkpoints)", "PASS")

## Cell 8: Quick Generation Test

In [ ]:
log.cell_start("Cell 8 — Quick Generation Test")

print("\n  Post-Injection Generation Test")
print("  " + "="*50)

# Move EVERYTHING to CPU (model.cpu() doesn't move nn.Parameter in field)
print("  Moving ALL model components to CPU...")

# Move main model
model.cpu()

# Explicitly move field tensors (they're nn.Parameters, not tracked by .cpu())
if hasattr(model.field, 'state'):
    model.field.state = model.field.state.cpu()
if hasattr(model.field, 'mass'):
    model.field.mass = model.field.mass.cpu()
if hasattr(model.field, 'energy'):
    model.field.energy = model.field.energy.cpu()

# Clear CUDA
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    # Verify GPU is clear
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"  ✓ Model moved to CPU (GPU: {allocated:.2f} GB used)")

test_prompts = [
    "The capital of France is",
    "User: Hello!\nAssistant:",
    "def fibonacci(n):",
    "Problem: If I have 5 apples and give away 2,",
    "The future of AI is",
]

model.eval()
with torch.no_grad():
    for prompt in test_prompts:
        try:
            response = model.generate(prompt, max_length=40, temperature=0.7)
            text = response.text if hasattr(response, 'text') else str(response)
            display_text = text[:70] + "..." if len(text) > 70 else text
            print(f'  "{prompt[:30]}..." → {display_text}')
        except Exception as e:
            print(f'  "{prompt[:30]}..." → ERROR: {e}')

print("\n  ✓ Generation test complete (model stays on CPU for save)")
log.cell_end("Cell 8 — Quick Generation Test", "PASS")

## Cell 9: Save Flux-capable.flx (FULL ARCHIVE)

This cell saves a COMPLETE .flx archive by:
1. Starting from the original Flux-X-complete.flx structure
2. Updating the field with injected knowledge
3. Preserving ALL 14 components

In [ ]:
log.cell_start("Cell 9 — Save Flux-capable.flx")

from datetime import datetime

print('\n  Building Flux-capable.flx (FULL ARCHIVE)...')
print('  ' + '='*50)

# CUDA recovery
if torch.cuda.is_available():
    try:
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print('  ✓ CUDA state cleared')
    except:
        pass

# Helper to move tensors to CPU
def safe_cpu(obj):
    """Recursively move tensors to CPU."""
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    elif isinstance(obj, dict):
        return {k: safe_cpu(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [safe_cpu(v) for v in obj]
    else:
        return obj

# Start from original .flx structure
print('  Starting from original Flux-X-complete.flx structure...')
flx_capable = safe_cpu(original_flx.copy())

# Update metadata
flx_capable['version'] = '1.5-capable'
flx_capable['metadata'] = {
    'source_phase': '9.7',
    'created': datetime.now().isoformat(),
    'description': 'FLUX with MASSIVE diverse dataset injection (500k+ samples)',
    'base_model': 'Flux-X-complete.flx',
    'injection': {
        'method': 'training_free_physics',
        'total_samples': total_injected,
        'unique_locations': unique_locations,
        'domains': {domain: stats['count'] for domain, stats in injection_stats.items()},
    },
}
print('  ✓ Metadata updated')

# Update field with ENRICHED state (the key change!)
print('  \n  Updating field with enriched knowledge...')
H, W, D, FEATURES = model.field.h, model.field.w, model.field.d, model.field.features

flx_capable['field'] = {
    'config': {'h': H, 'w': W, 'd': D, 'features': FEATURES},
    'state_dict': safe_cpu(model.field.state_dict()),
}

# Preserve gravity state from live model if available
if hasattr(model, 'gr'):
    try:
        flx_capable['field']['gravity_state'] = safe_cpu(model.gr.save_state())
        print('    ✓ Gravity state saved')
    except Exception as e:
        # Fall back to original
        if 'gravity_state' in original_flx.get('field', {}):
            flx_capable['field']['gravity_state'] = original_flx['field']['gravity_state']
            print('    ✓ Gravity state (from original)')

# Preserve thermodynamic state
if hasattr(model, 'tl'):
    try:
        flx_capable['field']['thermodynamic_state'] = safe_cpu(model.tl.save_state())
        print('    ✓ Thermodynamic state saved')
    except:
        if 'thermodynamic_state' in original_flx.get('field', {}):
            flx_capable['field']['thermodynamic_state'] = original_flx['field']['thermodynamic_state']
            print('    ✓ Thermodynamic state (from original)')

print('  ✓ Field updated (ENRICHED with massive knowledge)')

# CSE - use live model state
flx_capable['cse'] = {
    'config': original_flx.get('cse', {}).get('config', {'wave_dim': 432}),
    'state_dict': safe_cpu(model.cse.state_dict()),
}
print('  ✓ CSE')

# Decoder - use live model state (preserves all capabilities)
decoder_state = model.decoder.state_dict()
cleaned_decoder = {k.replace('_orig_mod.', ''): safe_cpu(v) for k, v in decoder_state.items()}
flx_capable['decoder'] = {
    'config': original_flx.get('decoder', {}).get('config', {'hidden_dim': 1024, 'num_layers': 4}),
    'state_dict': cleaned_decoder,
}
print('  ✓ WaveDecoder')

# Memory - preserve from original + update working memory
if 'memory' in original_flx:
    flx_capable['memory'] = safe_cpu(original_flx['memory'])
    print('  ✓ Memory (preserved)')

# Causal - preserve from original
if 'causal' in original_flx:
    flx_capable['causal'] = safe_cpu(original_flx['causal'])
    print('  ✓ Causal (preserved)')

# Bridges - use live model state
flx_capable['bridges'] = {}
if hasattr(model, 'wave_to_field'):
    flx_capable['bridges']['wave_to_field'] = safe_cpu(model.wave_to_field.state_dict())
if hasattr(model, 'field_to_wave'):
    flx_capable['bridges']['field_to_wave'] = safe_cpu(model.field_to_wave.state_dict())
if hasattr(model, 'output_head'):
    flx_capable['bridges']['output_head'] = safe_cpu(model.output_head.state_dict())
if 'router' in original_flx.get('bridges', {}):
    flx_capable['bridges']['router'] = original_flx['bridges']['router']
print('  ✓ Bridges')

# Adapters - preserve from original
if 'adapters' in original_flx:
    flx_capable['adapters'] = safe_cpu(original_flx['adapters'])
    print('  ✓ Adapters (preserved)')

# ─────────────────────────────────────────────
# Save
# ─────────────────────────────────────────────

output_path = CHECKPOINT_DIR / 'Flux-capable.flx'
torch.save(flx_capable, str(output_path))
size_mb = output_path.stat().st_size / 1e6

print(f'\n  ' + '='*50)
print(f'  ✓ Saved: {output_path.name} ({size_mb:.1f} MB)')
print(f'  ' + '='*50)

# Verify structure
print(f'\n  Archive structure:')
for key in flx_capable.keys():
    if isinstance(flx_capable[key], dict):
        subkeys = list(flx_capable[key].keys())[:5]
        print(f'    ✓ {key}: {subkeys}...')
    else:
        print(f'    ✓ {key}: {type(flx_capable[key]).__name__}')

log.metric("Output file", output_path.name)
log.metric("File size", f"{size_mb:.1f} MB")

log.success("Flux-capable.flx saved!")
log.cell_end("Cell 9 — Save Flux-capable.flx", "PASS")

## Cell 10: Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 10 — Upload to HuggingFace")

print('\n  Uploading Flux-capable.flx to HuggingFace Hub...')

from huggingface_hub import HfApi

if 'output_path' not in dir() or not output_path.exists():
    print('  ✗ No checkpoint to upload — run Cell 9 first')
    log.cell_end("Cell 10 — Upload to HuggingFace", "FAIL")
elif hf_token:
    try:
        api = HfApi(token=hf_token)
        
        api.upload_file(
            path_or_fileobj=str(output_path),
            path_in_repo='checkpoints/Flux-capable.flx',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'Add Flux-capable.flx (Phase 9.7 MASSIVE injection {total_injected:,} samples) — {datetime.now().isoformat()}',
        )
        
        print(f'  ✓ Uploaded to HuggingFace Hub')
        print(f'    https://huggingface.co/UnseenGAP/FLUX/blob/main/checkpoints/Flux-capable.flx')
        log.success("Flux-capable.flx uploaded!")
        log.cell_end("Cell 10 — Upload to HuggingFace", "PASS")
        
    except Exception as e:
        print(f'  ⚠ Upload failed: {e}')
        print(f'    File saved locally: {output_path}')
        log.cell_end("Cell 10 — Upload to HuggingFace", "WARN")
else:
    print('  ⚠ No HF_TOKEN - skipping upload')
    print(f'    File saved locally: {output_path}')
    log.cell_end("Cell 10 — Upload to HuggingFace", "SKIP")

## Cell 11: Summary

In [ ]:
log.separator("Phase 9.7 Complete: Diverse Dataset Injection")

print("""
╔════════════════════════════════════════════════════════════════════╗
║                    PHASE 9.7 COMPLETE                             ║
║             DIVERSE DATASET INJECTION → Flux-capable.flx          ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  DATASETS INJECTED (~155k samples - HALVED for memory):            ║
║  ├── SlimOrca (25k instruction following)                         ║
║  ├── OpenHermes (25k quality responses)                           ║
║  ├── UltraChat (15k quality conversations)                        ║
║  ├── MathInstruct (15k math reasoning)                            ║
║  ├── Evol-Instruct (15k complex instructions)                     ║
║  ├── FLAN (25k task templates)                                    ║
║  ├── + OpenAssistant, Dolly, GSM8K, Capybara, CodeAlpaca          ║
║                                                                    ║
║  MEMORY OPTIMIZATIONS:                                             ║
║  ✓ All counts halved (310k → 155k)                                ║
║  ✓ Memory clearing after each dataset                             ║
║  ✓ CPU-based settling when VRAM low                               ║
║                                                                    ║
║  KEY FEATURES:                                                     ║
║  ✓ Proper .flx structure preserved (all 14 components)            ║
║  ✓ Knowledge ADDED to existing field (not replaced)               ║
║  ✓ Full ~2GB checkpoint saved                                     ║
║                                                                    ║
║  THIS IS FLUX PHYSICS:                                             ║
║  ├── No loss functions                                            ║
║  ├── No gradients                                                  ║
║  ├── No epochs                                                     ║
║  └── Just physics: waves, fields, mass, energy                    ║
║                                                                    ║
║  OUTPUT: Flux-capable.flx                                          ║
╚════════════════════════════════════════════════════════════════════╝
""")

print("\nINJECTION STATISTICS:")
print(f"  Total samples injected: {total_injected:,}")
print(f"  Unique field locations: {unique_locations:,}")
H, W, D = model.field.h, model.field.w, model.field.d
print(f"  Coverage: {100*unique_locations/(H*W*D):.2f}% of field")
print(f"  Injection time: {total_time/60:.1f} minutes")
print()

print("DOMAINS INJECTED:")
for domain, stats in sorted(injection_stats.items(), key=lambda x: -x[1]['count']):
    print(f"  {domain:15s}: {stats['count']:>8,} samples")

log.success("Phase 9.7 complete!")